# Controlled contact-free trajectory comparison

**RQ3 follow-up: `T2_nocontact_traj`.** Compare full-FEM execution against the
switched FEM/FMU strategy at one NGSolve thread over 1.0 s. The default-thread
timing records remain separate evidence. This experiment tests repeatability
of each strategy at the actual comparison horizon before recording its
sampled trajectory separation.

The 0.085 rad threshold and 0.015 rad band match `04_performance`, rather than
the different defaults in `NO_CONTACT_SCENARIO`. The controller, quantized
sensor, FMU exports, physical parameters, and event tolerances are unchanged.
Both cases use `HybridAlgorithm`; the full-FEM case has the same constant
detection probe as the timing campaign.

There are two executions per case, each in a fresh process, with no warm-up.
These are reproducibility checks, not timing repetitions. The coordinator and
workers pin NGSolve before initialization. The notebook kernel's thread setting
is untouched because it never imports NGSolve.

**Before the production run:** commit and push this notebook and
`evidence/trajectory.py` in the SysSimX repository. The producer refuses dirty
measurement code or a revision absent from `origin/main` before it launches a
physical run. Do not edit measured code while a campaign is running.

In [ ]:
import json

import pandas as pd

from evidence import repo_root

REPO = repo_root()
from evidence.trajectory import make_scenario, run_experiment
from record import paper_results_dir

SMOKE = False
RESULTS, destination = paper_results_dir(REPO / "notebooks")
SCENARIO = make_scenario(smoke=SMOKE)
print("Results:", RESULTS, "(" + destination + ")")
display(pd.Series(SCENARIO.provenance()).to_frame("value"))

## Execute and validate

The production campaign is four physical runs. Every worker saves its native
time grid, angle, angular velocity, sensor output, decoded angle, controller
command, drive torque, switch identities and times, and event history. Its
metadata records the source revision, backend versions, and hashes of the
actual FMU inputs and raw archive.

`SMOKE = True` uses 0.05 s and writes only `T2_nocontact_traj.smoke.json`.
It is a pipeline check and need not exercise switching. It cannot replace the
1.0 s production experiment.

The aggregate refuses non-finite traces, incomplete horizons, contacts,
reported missed events, mismatched code or FMU inputs, and nonidentical
repetitions. Both production switched runs must actually switch. Failure keeps
the raw runs for diagnosis and writes no new campaign record.

In [ ]:
artifact = run_experiment(RESULTS, smoke=SMOKE)
document = json.loads(artifact.read_text(encoding="utf-8"))
values = document["values"]
print("Recorded:", artifact)
print("Raw runs:", values["raw_directory"])
display(pd.DataFrame(values["per_pair"]))
display(pd.Series({
    case: result["bit_identical"]
    for case, result in values["repeatability"].items()
}).to_frame("bit_identical"))

## How the comparison may be used

The metric uses the union of both recorded plant time grids, with linear
interpolation and no extrapolation. Repeated timestamps use the last sample.
Both traces include the initial state, so the RMS separation integrates over
the declared horizon. It is a sampled numerical comparison, not a continuous
error bound or physical validation.

The result describes the combined difference between two execution strategies,
including model representations, handover initialization, solver restarts,
event handling, and closed-loop feedback. Repeatability does not isolate an
error caused solely by switching. The diagnostic controller and sensor signals
support investigating quantization amplification; they do not establish that
cause automatically.

Do not combine this single-thread deviation with the default-thread speedup as
one measured accuracy/runtime operating point. Once the production record is
admitted, update the paper's thread protocol and §6.3, then §7 and the abstract.
The existing default-thread trajectory pairs remain descriptive observations.

The same producer is available from `notebooks/` without Jupyter:

```text
python -m evidence.trajectory --preflight
python -m evidence.trajectory
```

Archive the complete directory printed above with the compact JSON record.
It contains four `.npz` files, worker metadata and logs, and `manifest.json`.